# 4.4 Lab: Data Prep & Feature Engineering

## Table of Contents
- [4.4.1 Setup and Data Overview](#441-setup-and-data-overview)
- [4.4.2 Missing Value Imputation](#442-missing-value-imputation)
- [4.4.3 Feature Scaling](#443-feature-scaling)
- [4.4.4 Categorical Encoding](#444-categorical-encoding)
- [4.4.5 Feature Engineering](#445-feature-engineering)
- [4.4.6 Model Training with Preprocessed Data](#446-model-training-with-preprocessed-data)
- [Knowledge Check](#knowledge-check)
- [Mini-Challenges](#mini-challenges)
- [Practical Connections](#practical-connections)

## Introduction

In this lab, we'll practice applying all the data preparation and feature engineering techniques we've learned. We'll work with a dataset, implement missing value imputation, feature scaling, categorical encoding, and feature engineering, and then build a model with our preprocessed data.

First, let's import the necessary libraries:

In [ ]:
python
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, 
    classification_report, 
    accuracy_score, 
    roc_curve, 
    roc_auc_score
)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Ensure reproducibility
np.random.seed(42)

## 4.4.1 Setup and Data Overview

Let's start by loading the Breast Cancer dataset and examining its structure:

In [ ]:
python
# Load the Breast Cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Convert to DataFrames for easier manipulation
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)
y_train_df = pd.Series(y_train, name='target')
y_test_df = pd.Series(y_test, name='target')

# Display dataset information
print("Breast Cancer Dataset")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Target classes: {cancer.target_names}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# Display the first few rows of the training set
print("\nFirst 5 rows of the training data:")
print(X_train_df.head())

# Check basic statistics
print("\nBasic statistics of the training data:")
print(X_train_df.describe().round(3))

# Check for missing values
missing_values = X_train_df.isnull().sum()
if missing_values.sum() > 0:
    print("\nMissing values per feature:")
    print(missing_values[missing_values > 0])
else:
    print("\nNo missing values found in the dataset.")

The Breast Cancer dataset from sklearn doesn't have missing values. Let's artificially introduce some to practice imputation techniques:

In [ ]:
python
# Artificially introduce missing values into selected features of the training set
np.random.seed(42)

# Select features where we'll introduce missing values
features_to_modify = ['mean radius', 'mean texture', 'mean perimeter']

# Introduce about 10% missing values in selected features
for feature in features_to_modify:
    # Randomly select indices where values will be missing
    missing_indices = np.random.choice(
        X_train_df.index, 
        size=int(0.1 * len(X_train_df)), 
        replace=False
    )
    # Set values to NaN at selected indices
    X_train_df.loc[missing_indices, feature] = np.nan

# Check for missing values after introduction
missing_values = X_train_df.isnull().sum()
print("\nMissing values per feature after artificial introduction:")
print(missing_values[missing_values > 0])

# Visualize missing values
plt.figure(figsize=(12, 6))
sns.heatmap(X_train_df.isnull(), cmap='viridis', cbar=False, yticklabels=False)
plt.title('Missing Values in Training Data')
plt.xlabel('Features')
plt.ylabel('Samples')
plt.show()

## 4.4.2 Missing Value Imputation

Now, let's handle these missing values using various imputation techniques. We'll compare mean, median, and most frequent value imputation:

In [ ]:
python
# 1. Mean Imputation
mean_imputer = SimpleImputer(strategy='mean')
X_train_mean_imputed = mean_imputer.fit_transform(X_train_df)
X_test_mean_imputed = mean_imputer.transform(X_test_df)

# Convert back to DataFrame for easier analysis
X_train_mean_df = pd.DataFrame(X_train_mean_imputed, columns=feature_names, index=X_train_df.index)
X_test_mean_df = pd.DataFrame(X_test_mean_imputed, columns=feature_names, index=X_test_df.index)

# 2. Median Imputation
median_imputer = SimpleImputer(strategy='median')
X_train_median_imputed = median_imputer.fit_transform(X_train_df)
X_test_median_imputed = median_imputer.transform(X_test_df)

# Convert back to DataFrame
X_train_median_df = pd.DataFrame(X_train_median_imputed, columns=feature_names, index=X_train_df.index)
X_test_median_df = pd.DataFrame(X_test_median_imputed, columns=feature_names, index=X_test_df.index)

# 3. Most Frequent Value Imputation
mode_imputer = SimpleImputer(strategy='most_frequent')
X_train_mode_imputed = mode_imputer.fit_transform(X_train_df)
X_test_mode_imputed = mode_imputer.transform(X_test_df)

# Convert back to DataFrame
X_train_mode_df = pd.DataFrame(X_train_mode_imputed, columns=feature_names, index=X_train_df.index)
X_test_mode_df = pd.DataFrame(X_test_mode_imputed, columns=feature_names, index=X_test_df.index)

# Compare the imputation values used for each method
imputation_values = pd.DataFrame({
    'Original Mean': X_train_df.mean(),
    'Mean Imputation': mean_imputer.statistics_,
    'Median Imputation': median_imputer.statistics_,
    'Most Frequent Imputation': mode_imputer.statistics_
})

print("Imputation values used for missing features:")
print(imputation_values.loc[features_to_modify].round(3))

# Visualize distributions before and after imputation for an impacted feature
feature_to_plot = 'mean radius'

plt.figure(figsize=(15, 5))

# Original distribution (non-missing values only)
plt.subplot(1, 4, 1)
sns.histplot(X_train_df[feature_to_plot].dropna(), kde=True, color='blue')
plt.title('Original (Non-Missing)')
plt.xlabel(feature_to_plot)

# Mean imputation
plt.subplot(1, 4, 2)
sns.histplot(X_train_mean_df[feature_to_plot], kde=True, color='green')
plt.title('Mean Imputation')
plt.xlabel(feature_to_plot)

# Median imputation
plt.subplot(1, 4, 3)
sns.histplot(X_train_median_df[feature_to_plot], kde=True, color='orange')
plt.title('Median Imputation')
plt.xlabel(feature_to_plot)

# Mode imputation
plt.subplot(1, 4, 4)
sns.histplot(X_train_mode_df[feature_to_plot], kde=True, color='red')
plt.title('Most Frequent Imputation')
plt.xlabel(feature_to_plot)

plt.tight_layout()
plt.show()

# For simplicity, let's proceed with median imputation for the rest of this lab
X_train_imputed = X_train_median_df
X_test_imputed = X_test_median_df

## 4.4.3 Feature Scaling

Now, let's apply feature scaling to our imputed data:

In [ ]:
python
# Apply StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Convert back to DataFrame for easier viewing
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train_imputed.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_names, index=X_test_imputed.index)

# Display the first few rows of the scaled data
print("First 5 rows of scaled training data:")
print(X_train_scaled_df.head().round(3))

# Check the distribution of the scaled data
print("\nScaled data statistics:")
print(X_train_scaled_df.describe().round(3))

# Visualize the effect of scaling on selected features
selected_features = ['mean radius', 'mean texture', 'mean perimeter']

plt.figure(figsize=(15, 10))

# Plot original distributions
for i, feature in enumerate(selected_features):
    plt.subplot(2, 3, i+1)
    sns.histplot(X_train_imputed[feature], kde=True, color='blue')
    plt.title(f'Original: {feature}')
    plt.xlabel(feature)

# Plot scaled distributions
for i, feature in enumerate(selected_features):
    plt.subplot(2, 3, i+4)
    sns.histplot(X_train_scaled_df[feature], kde=True, color='green')
    plt.title(f'Scaled: {feature}')
    plt.xlabel(f'{feature} (Standardized)')

plt.tight_layout()
plt.show()

# Visualize feature correlations before and after scaling
plt.figure(figsize=(15, 7))

# Original data correlations
plt.subplot(1, 2, 1)
correlation_matrix = X_train_imputed[selected_features].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlations Before Scaling')

# Scaled data correlations
plt.subplot(1, 2, 2)
correlation_matrix_scaled = X_train_scaled_df[selected_features].corr()
sns.heatmap(correlation_matrix_scaled, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlations After Scaling')

plt.tight_layout()
plt.show()

## 4.4.4 Categorical Encoding

The Breast Cancer dataset doesn't have categorical features. Let's add a synthetic categorical feature to demonstrate encoding techniques:

In [ ]:
python
# Create a synthetic categorical feature
np.random.seed(0)

# Add a categorical feature to train and test sets
X_train_scaled_df['tumor_location'] = np.random.choice(
    ['Upper-Outer', 'Upper-Inner', 'Lower-Outer', 'Lower-Inner', 'Central'], 
    size=len(X_train_scaled_df)
)

X_test_scaled_df['tumor_location'] = np.random.choice(
    ['Upper-Outer', 'Upper-Inner', 'Lower-Outer', 'Lower-Inner', 'Central'], 
    size=len(X_test_scaled_df)
)

# Display the first few rows with the added categorical feature
print("Data with added categorical feature:")
print(X_train_scaled_df.head())

# Check the distribution of the categorical feature
plt.figure(figsize=(10, 5))
sns.countplot(x='tumor_location', data=X_train_scaled_df)
plt.title('Distribution of Tumor Location (Categorical Feature)')
plt.xlabel('Tumor Location')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

# 1. One-Hot Encoding
# Create a copy of the data before applying one-hot encoding
X_train_with_cat = X_train_scaled_df.copy()
X_test_with_cat = X_test_scaled_df.copy()

# Apply one-hot encoding
X_train_onehot = pd.get_dummies(X_train_with_cat, columns=['tumor_location'], drop_first=True)
X_test_onehot = pd.get_dummies(X_test_with_cat, columns=['tumor_location'], drop_first=True)

# Ensure that test set has the same columns as training set
for col in X_train_onehot.columns:
    if col not in X_test_onehot.columns:
        X_test_onehot[col] = 0

# Reorder test set columns to match training set
X_test_onehot = X_test_onehot[X_train_onehot.columns]

# Display the result of one-hot encoding
print("\nAfter One-Hot Encoding (first few rows, last few columns):")
print(X_train_onehot.iloc[:5, -5:])
print(f"\nShape after one-hot encoding: {X_train_onehot.shape}")

# 2. Label Encoding
# Create a copy of the data
X_train_for_label = X_train_scaled_df.copy()
X_test_for_label = X_test_scaled_df.copy()

# Apply label encoding
label_encoder = LabelEncoder()
X_train_for_label['tumor_location_encoded'] = label_encoder.fit_transform(X_train_for_label['tumor_location'])
X_test_for_label['tumor_location_encoded'] = label_encoder.transform(X_test_for_label['tumor_location'])

# Display the result of label encoding
print("\nAfter Label Encoding:")
print(X_train_for_label[['tumor_location', 'tumor_location_encoded']].head())
print("\nLabel encoding mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")

# We'll proceed with the one-hot encoded data for the rest of the lab
X_train_processed = X_train_onehot
X_test_processed = X_test_onehot

## 4.4.5 Feature Engineering

Now, let's create some new features based on the existing ones:

In [ ]:
python
# Original numerical features (before adding categorical)
orig_features = feature_names.tolist()

# 1. Create interaction features
# Interaction between radius and perimeter
X_train_processed['radius_perimeter_ratio'] = X_train_imputed['mean radius'] / X_train_imputed['mean perimeter']
X_test_processed['radius_perimeter_ratio'] = X_test_imputed['mean radius'] / X_test_imputed['mean perimeter']

# Interaction between texture and smoothness
X_train_processed['texture_smoothness_product'] = X_train_imputed['mean texture'] * X_train_imputed['mean smoothness']
X_test_processed['texture_smoothness_product'] = X_test_imputed['mean texture'] * X_test_imputed['mean smoothness']

# 2. Create aggregate features
# Average of the mean texture, mean smoothness, and mean compactness
X_train_processed['texture_smoothness_compactness_avg'] = X_train_imputed[['mean texture', 'mean smoothness', 'mean compactness']].mean(axis=1)
X_test_processed['texture_smoothness_compactness_avg'] = X_test_imputed[['mean texture', 'mean smoothness', 'mean compactness']].mean(axis=1)

# 3. Create polynomial features for radius
X_train_processed['radius_squared'] = X_train_imputed['mean radius'] ** 2
X_test_processed['radius_squared'] = X_test_imputed['mean radius'] ** 2

# Display the data with engineered features
print("First few rows with engineered features:")
print(X_train_processed.iloc[:5, -5:])

# Visualize the relationship between engineered features and the target
# Combine with target for visualization
train_with_target = pd.concat([X_train_processed, y_train_df], axis=1)

plt.figure(figsize=(15, 10))

# Plot relationships for engineered features
engineered_features = [
    'radius_perimeter_ratio', 
    'texture_smoothness_product', 
    'texture_smoothness_compactness_avg', 
    'radius_squared'
]

for i, feature in enumerate(engineered_features):
    plt.subplot(2, 2, i+1)
    sns.boxplot(x='target', y=feature, data=train_with_target)
    plt.title(f'{feature} vs Target')
    plt.xlabel('Target (0=Malignant, 1=Benign)')
    plt.ylabel(feature)

plt.tight_layout()
plt.show()

# Calculate feature correlations with target
correlations = pd.DataFrame({
    'Feature': X_train_processed.columns,
    'Correlation with Target': [np.corrcoef(X_train_processed[col], y_train_df)[0, 1] for col in X_train_processed.columns]
})

# Sort by absolute correlation
correlations['Abs Correlation'] = abs(correlations['Correlation with Target'])
correlations = correlations.sort_values('Abs Correlation', ascending=False)

print("\nTop 10 features by correlation with target:")
print(correlations.head(10).round(3))

# Visualize top feature correlations
plt.figure(figsize=(12, 6))
top_10_features = correlations.head(10)['Feature'].values
sns.barplot(x='Correlation with Target', y='Feature', data=correlations.head(10))
plt.title('Top 10 Features by Correlation with Target')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Feature')
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
plt.show()

## 4.4.6 Model Training with Preprocessed Data

Finally, let's train a model using our preprocessed data and evaluate its performance:

In [ ]:
python
# Train a logistic regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_processed, y_train_df)

# Make predictions
y_pred = log_reg.predict(X_test_processed)
y_pred_proba = log_reg.predict_proba(X_test_processed)[:, 1]  # Probability of positive class

# Evaluate the model
accuracy = accuracy_score(y_test_df, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

# Create a confusion matrix
cm = confusion_matrix(y_test_df, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=cancer.target_names, 
            yticklabels=cancer.target_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Print a classification report
print("\nClassification Report:")
print(classification_report(y_test_df, y_pred, target_names=cancer.target_names))

# Plot ROC curve
fpr, tpr, thresholds = roc_curve(y_test_df, y_pred_proba)
auc_score = roc_auc_score(y_test_df, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

# Comparison with a model trained on just the scaled data without feature engineering
# Use the data after imputation and scaling but before adding categorical and engineered features
X_train_simple = X_train_scaled_df.drop(columns=['tumor_location'])
X_test_simple = X_test_scaled_df.drop(columns=['tumor_location'])

# Train another logistic regression model
log_reg_simple = LogisticRegression(random_state=42, max_iter=1000)
log_reg_simple.fit(X_train_simple, y_train_df)

# Make predictions
y_pred_simple = log_reg_simple.predict(X_test_simple)
y_pred_proba_simple = log_reg_simple.predict_proba(X_test_simple)[:, 1]

# Evaluate the simple model
accuracy_simple = accuracy_score(y_test_df, y_pred_simple)
auc_score_simple = roc_auc_score(y_test_df, y_pred_proba_simple)

# Compare the models
print("\nModel Comparison:")
print(f"Full Preprocessing Pipeline - Accuracy: {accuracy:.4f}, AUC: {auc_score:.4f}")
print(f"Simple Scaling Only - Accuracy: {accuracy_simple:.4f}, AUC: {auc_score_simple:.4f}")
print(f"Improvement - Accuracy: {accuracy - accuracy_simple:.4f}, AUC: {auc_score - auc_score_simple:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X_train_processed.columns,
    'Coefficient': np.abs(log_reg.coef_[0])
})
feature_importance = feature_importance.sort_values('Coefficient', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x='Coefficient', y='Feature', data=feature_importance.head(10))
plt.title('Top 10 Most Important Features (Logistic Regression Coefficients)')
plt.xlabel('Absolute Coefficient Value')
plt.ylabel('Feature')
plt.show()

# Try a different model for comparison (Random Forest)
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train_processed, y_train_df)

# Make predictions
y_pred_rf = rf_model.predict(X_test_processed)
y_pred_proba_rf = rf_model.predict_proba(X_test_processed)[:, 1]

# Evaluate the Random Forest model
accuracy_rf = accuracy_score(y_test_df, y_pred_rf)
auc_score_rf = roc_auc_score(y_test_df, y_pred_proba_rf)

print("\nRandom Forest Model:")
print(f"Accuracy: {accuracy_rf:.4f}, AUC: {auc_score_rf:.4f}")

# Compare all models
models = ['Logistic Regression (Full Pipeline)', 'Logistic Regression (Simple)', 'Random Forest']
accuracies = [accuracy, accuracy_simple, accuracy_rf]
auc_scores = [auc_score, auc_score_simple, auc_score_rf]

plt.figure(figsize=(12, 5))
x = np.arange(len(models))
width = 0.35

plt.bar(x - width/2, accuracies, width, label='Accuracy')
plt.bar(x + width/2, auc_scores, width, label='AUC Score')

plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.xticks(x, models, rotation=15)
plt.legend()
plt.tight_layout()
plt.show()

## Lab Conclusions

This lab has taken us through the entire data preprocessing and feature engineering pipeline, from handling missing values to scaling, encoding categorical features, and creating new features. We've seen how each step can be implemented and evaluated its impact on model performance.

Key takeaways:
1. Proper imputation of missing values preserves data that would otherwise be lost
2. Scaling features to a similar range is essential for many algorithms
3. Categorical features need to be encoded appropriately
4. Feature engineering can create new predictors that capture complex relationships
5. The combination of all these techniques can significantly improve model performance

In this case, we saw that our full preprocessing pipeline led to improved performance compared to simple scaling alone, demonstrating the value of comprehensive data preparation.

## Knowledge Check

1. Why should we fit imputers and scalers only on the training data?
2. What's the potential impact of not handling missing values before model training?
3. How does feature scaling affect distance-based algorithms like KNN?
4. When is one-hot encoding preferred over label encoding?
5. What types of feature interactions are most likely to improve model performance?

## Mini-Challenges

### Challenge 1: Complete Preprocessing Pipeline
Create a scikit-learn Pipeline that performs all the preprocessing steps we did manually in this lab:
1. Missing value imputation
2. Feature scaling
3. Categorical encoding

In [ ]:
python
# Starter code
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Your code to create a scikit-learn pipeline

### Challenge 2: Feature Selection
Implement a feature selection technique to identify and remove less important features from our preprocessed data. Then train a model with the selected features and compare its performance with our full-feature model.

In [ ]:
python
# Starter code
from sklearn.feature_selection import SelectKBest, f_classif, RFE

# Your code to implement feature selection

### Challenge 3: Cross-Validation Comparison
Implement k-fold cross-validation to more robustly compare the performance of models trained with different preprocessing approaches.

In [ ]:
python
# Starter code
from sklearn.model_selection import cross_val_score, KFold

# Your code to implement cross-validation evaluation

## Practical Connections

- **Healthcare**: In medical datasets, missing values often contain meaningful information (e.g., tests not ordered because the doctor didn't think they were necessary), requiring careful handling during preprocessing.

- **Finance**: In credit scoring models, feature engineering often creates ratios (e.g., debt-to-income) that are more predictive than the raw values.

- **Marketing**: Customer segmentation models require appropriate scaling of features to prevent variables with larger ranges (like income) from dominating those with smaller ranges (like age).

- **Retail**: When analyzing product data, categorical features like brand or category need appropriate encoding to be useful in predictive models.

- **Manufacturing**: Quality control models often benefit from polynomial features that can capture non-linear relationships between production parameters and defect rates.

- **HR Analytics**: Employee attrition models require handling imbalanced data (as most employees don't leave) alongside careful preprocessing of categorical features like departments or job roles.